# VCD on POPE — Colab runner (A100)

**Budget rules:** Colab bills compute units for every minute a runtime is connected, *including idle time*.
Run the setup cells, start one phase, and leave the tab open; the phase cells **disconnect the runtime automatically** when they finish.
Never leave a runtime connected while you are not running something.

Results and images live on Google Drive, so every phase can be resumed in a new session: re-run sections 1–2, then the phase cell.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Setup (~3 min). After the pip cell: *Runtime → Restart session* once, then re-run section 1 from the top.

In [ ]:
REPO_URL = "https://github.com/mirhanayd/yzm5110-vcd-pope.git"
DRIVE = "/content/drive/MyDrive/yzm5110_vcd_results"

from google.colab import drive
drive.mount('/content/drive')
import os
for d in ("runs", "timing", "images", "logs", "jobs"):
    os.makedirs(f"{DRIVE}/{d}", exist_ok=True)
if not os.path.exists("/content/vcd-pope"):
    !git clone $REPO_URL /content/vcd-pope
%cd /content/vcd-pope
!git pull --ff-only
# keep all outputs (and the downloaded COCO images) on Drive
!mkdir -p results data && for d in runs timing logs jobs; do rm -rf results/$d; ln -sfn $DRIVE/$d results/$d; done
!rm -rf data/images && ln -sfn $DRIVE/images data/images
!git rev-parse HEAD

In [ ]:
!pip install -q -r requirements.txt 2>&1 | grep -v "dependency conflicts\|requires huggingface-hub" || true
import transformers; print(transformers.__version__)   # must print 4.51.3, otherwise restart the session

## 2. Data (first session only downloads; later sessions find the images on Drive)

In [ ]:
!python scripts/download_data.py

## 3. Phase 1 — gate tuning on dev + reproduction with controls (~2–2.5 h on A100, 4 workers)
Progress: `!tail -n 3 results/logs/phase1_worker*.log` in another cell. If the session dies, just run sections 1–2 and this cell again.

In [ ]:
!WORKERS=4 bash scripts/experiments.sh phase1
!python scripts/make_tables.py
from google.colab import runtime; runtime.unassign()   # stop billing

## 4. Phase 2 — improvement on COCO + clean timing runs (~1.5 h). Set TAU and SKIP from the output of `pick_tau.py`.

In [ ]:
!python scripts/pick_tau.py

In [ ]:
TAU = "0.9"     # <-- from pick_tau.py
SKIP = "0.5"    # <-- from pick_tau.py
!TAU=$TAU SKIP=$SKIP WORKERS=4 bash scripts/experiments.sh phase2
!bash scripts/experiments.sh tables
from google.colab import runtime; runtime.unassign()   # stop billing

## 5. Tables (no GPU needed: can run on a CPU runtime)

In [ ]:
!bash scripts/experiments.sh tables
!cat results/tables/main.md results/tables/timing_main.md